# Resume training the 23.4M OrbitNet BC from epoch-2 (A100)

Continues the Stage-B 23.4M gate-head BC (`640/L6/ff1280/h8`) **from the epoch-2 checkpoint**,
saving a checkpoint **every epoch** to Drive (rolling `.last.pt` + a per-epoch archive
`ep_NNN.pt`), and includes an **arena-gating cell** to rank the epochs.

**Why resume:** the 23.4M was *undertrained* — arena win-rate vs producer climbed every epoch
(17.5% → 33% → 40%) and by epoch 2 nearly matched the 15M's peak (45%), still rising.

**⚠️ Data note:** the original 26M-state Lambda shards are gone (instance terminated). This
resumes on the **12.8M-state shards on Drive** (`MyDrive/orbit_wars/shards`) — same top-ladder
win-weighted BC regime, smaller corpus. Model + optimizer continue fine. For the full corpus,
rebuild shards with the harvest cells in `winbc_colab.ipynb` and point `SHARDS` there.

**⚠️ Gate by ARENA, not val-`sel`:** best-on-`sel` diverged from arena and the arena win-rate
OSCILLATES (the 15M went 28→45→8→37). This notebook archives **every** epoch and the gating
cell ranks them — keep the epoch with the highest win-rate vs producer (not last, not best-sel).

**Disconnect-proof:** every cell writes to Drive. If Colab drops, re-run all cells — training
resumes from the latest `.last.pt` automatically.


In [ ]:
# === Cell 1: Setup — mount Drive, clone/pull repo, install, GPU check ===
import os, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/orbit_wars'
SHARDS = f'{DRIVE}/shards'
CKPTS  = f'{DRIVE}/checkpoints'
for d in (DRIVE, SHARDS, CKPTS):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'   # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.system('pip install -q torch numpy pyyaml')   # training needs no kaggle-environments
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('CUDA:', torch.cuda.is_available(), name)
os.system('df -h /content | tail -1')


In [ ]:
# === Cell 2: Stage shards to fast local SSD ===
import os, json
SHARDS_LOCAL = '/content/shards'
os.makedirs(SHARDS_LOCAL, exist_ok=True)
if os.path.exists(f'{SHARDS_LOCAL}/manifest.json'):
    print('shards already staged on local /content')
elif os.path.exists(f'{SHARDS}/manifest.json'):
    print('staging Drive shards -> local /content (one-time copy)...')
    os.system(f'cp -r {SHARDS}/* {SHARDS_LOCAL}/')
    print('done')
else:
    raise SystemExit(f'No shards at {SHARDS}. Build them first (winbc_colab.ipynb harvest cells).')
m = json.load(open(f'{SHARDS_LOCAL}/manifest.json'))
print(f"shards: {m['n_examples']:,} examples, {m['n_shards']} files, w_loss={m['w_loss']}")


In [ ]:
# === Cell 3: Seed the resume from the epoch-2 checkpoint (first run only) ===
import os, shutil, torch
OUT     = f'{CKPTS}/winbc_24m_resume.pt'
LAST    = f'{CKPTS}/winbc_24m_resume.last.pt'
CURVES  = f'{CKPTS}/winbc_24m_resume_curves.csv'
ARCHDIR = f'{CKPTS}/winbc_24m_resume_epochs'   # per-epoch ep_NNN.pt land here (arena-gate each)
SEED    = f'{CKPTS}/winbc_24m_ep2.pt'          # the epoch-2 checkpoint (already on Drive)

if os.path.exists(LAST):
    ep = torch.load(LAST, map_location='cpu', weights_only=False).get('epoch')
    print(f'Found {os.path.basename(LAST)} (epoch {ep}) -> RESUMING from there. Not re-seeding.')
elif os.path.exists(SEED):
    shutil.copy(SEED, LAST)
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    print(f"Seeded from epoch-2 (epoch={ck.get('epoch')}, best sel={ck.get('best'):.3f}, arch={ck.get('arch')}).")
    print(f'Training will CONTINUE from epoch {int(ck.get("epoch"))+1}.')
else:
    raise SystemExit(f'Missing both {LAST} and {SEED}. Upload winbc_24m_ep2.pt to {CKPTS}/ first.')


In [ ]:
# === Cell 4: Train (resume the 23.4M; save EVERY epoch to Drive) ===
# Arch MUST match the ckpt: 640/L6/ff1280/h8. Batch 512 fits A100-40GB (~20GB). The loaded
# optimizer restores the original LR (~4e-4) so we just continue the trajectory. Each epoch
# saves best-on-sel -> {OUT}, rolling .last.pt -> {LAST}, per-epoch -> {ARCHDIR}/ep_NNN.pt.
# Re-run after any disconnect; it resumes from the latest .last.pt on Drive.
EPOCHS = 20   # ceiling; with per-epoch archiving you can stop & gate anytime
BATCH  = 512  # reduce to 384/256 if a non-40GB GPU OOMs
!python scripts/winbc_train.py --shards {SHARDS_LOCAL} --gate_head --resume \
    --embed-dim 640 --n-layers 6 --ff-dim 1280 --n-heads 8 \
    --epochs {EPOCHS} --batch {BATCH} --lr 4e-4 --launch_weight 8.0 --w_loss 0.3 \
    --shard-cache auto \
    --out {OUT} --curves {CURVES} --archive-dir {ARCHDIR}


In [ ]:
# === Cell 5: Arena-gate each archived epoch vs producer (find the PEAK) ===
# Arena win-rate oscillates & diverges from val-sel, so rank epochs by the REAL metric. Runs
# the faithful bc_teacher executor (gate->pointer->capture, min/0.5) vs producer for each
# ep_NNN.pt. CPU-bound (producer is slow): ~a few min/epoch at n=120 on Colab's vCPUs.
import os, sys, glob, csv, subprocess
os.system('pip install -q "kaggle-environments>=1.28.0"')   # only needed for gating

GATE_GAMES   = 120                 # >=100 per the project noise-floor rule (lower to scan faster)
GATE_WORKERS = max(2, (os.cpu_count() or 4) - 1)
OPPONENTS    = 'producer'          # set to 'producer,v5' to also gate vs v5 (slower)

def winrates(csv_path):
    wr = {}
    for r in csv.DictReader(open(csv_path)):
        a, b = r['agent_a'], r['agent_b']
        if 'bc_teacher' not in (a, b):
            continue
        o = b if a == 'bc_teacher' else a
        ra = float(r['reward_a']) if a == 'bc_teacher' else float(r['reward_b'])
        d = wr.setdefault(o, [0.0, 0])
        d[0] += 1.0 if ra > 0 else (0.5 if ra == 0 else 0.0); d[1] += 1
    return {o: s / n for o, (s, n) in wr.items()}

eps = sorted(glob.glob(f'{ARCHDIR}/ep_*.pt'))
print(f'gating {len(eps)} epoch(s) vs {OPPONENTS} at n={GATE_GAMES} ({GATE_WORKERS} workers)...\n')
results = []
for ck in eps:
    name = os.path.basename(ck)
    out_csv = f'/content/gate_{name}.csv'
    if os.path.exists(out_csv):
        os.remove(out_csv)   # fresh gate each run
    env = {**os.environ, 'OW_BC_TEACHER_CKPT': ck, 'OW_BC_DRAIN': 'min', 'OW_BC_GATE_THR': '0.5'}
    subprocess.run([sys.executable, 'scripts/arena.py', '--agents', f'bc_teacher,{OPPONENTS}',
                    '--games', str(GATE_GAMES), '--workers', str(GATE_WORKERS), '--out', out_csv],
                   env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    wr = winrates(out_csv)
    results.append((name, wr))
    print(f'  {name}: ' + '  '.join(f'vs {o} {v:.1%}' for o, v in wr.items()))

print('\n=== RANKED by win-rate vs producer (keep the top one) ===')
for name, wr in sorted(results, key=lambda x: -x[1].get('producer', 0)):
    print(f'  {name}: ' + '  '.join(f'vs {o} {v:.1%}' for o, v in wr.items()))


## Ship the peak epoch

Take the top `ep_NNN.pt` from Cell 5's ranking and build a verified submission bundle from it
(locally or on a box with the repo):

```bash
uv run python scripts/build_bc_bundle.py --ckpt ep_NNN.pt --drain min --gate-thr 0.5
```

That assembles `outputs/submissions/bc_bundle.tar.gz` and verifies it through Kaggle's loader.

Reminder: trust the **arena ranking** (Cell 5), not best-on-`sel` (`winbc_24m_resume.pt`) or the
last epoch — arena win-rate oscillates and only weakly tracks val-`sel`.
